# Aggregate Model Metrics From JSON

Load evaluation JSON files from multiple result directories, extract per-dataset metrics for selected model(s), build per-model tables, and save one concatenated table.

In [ ]:
from __future__ import annotations

import json
from pathlib import Path
from typing import Any

import numpy as np
import pandas as pd
from IPython.display import display

pd.set_option("display.max_columns", 80)
pd.set_option("display.width", 220)

## Parameters

In [ ]:
REPO_ROOT = Path.cwd().resolve().parents[2] if Path.cwd().name == "evaluation" else Path.cwd().resolve()

EVAL_DIR = REPO_ROOT / "examples" / "pooling" / "evaluation" / "results"

# Provide one or many result directories.
RESULT_SOURCES = [
     EVAL_DIR / "nanobeir_selection/nanobeir_pooling_sweep_20260215_195731",
     EVAL_DIR / "scifact-pool-none/nanobeir_selection/nanobeir_pooling_sweep_20260219_153737",
     EVAL_DIR / "scifact-pool-span-2/nanobeir_selection/nanobeir_pooling_sweep_20260217_220054",
     EVAL_DIR / "scifact-pool-span-3/nanobeir_selection/nanobeir_pooling_sweep_20260218_183601",
     EVAL_DIR / "scifact-pool-span-all/nanobeir_selection/nanobeir_pooling_sweep_20260218_200822",
     EVAL_DIR / "scifact-pool-span-all-v2/nanobeir_selection/nanobeir_pooling_sweep_20260221_164408",
     EVAL_DIR / "scifact-pool-hier-2/nanobeir_selection/nanobeir_pooling_sweep_20260219_165230",
     EVAL_DIR / "scifact-pool-hier-all/nanobeir_selection/nanobeir_pooling_sweep_20260218_202148",
     EVAL_DIR / "scifact-pool-hier-all-v2/nanobeir_selection/nanobeir_pooling_sweep_20260221_142637",
     EVAL_DIR / "scifact-pool-kmeans-2/nanobeir_selection/nanobeir_pooling_sweep_20260219_180444",
     EVAL_DIR / "scifact-pool-kmeans-all/nanobeir_selection/nanobeir_pooling_sweep_20260218_213421",
     EVAL_DIR / "scifact-pool-kmeans-all-v2/nanobeir_selection/nanobeir_pooling_sweep_20260221_113955",
     EVAL_DIR / "fiqa-pool-none/nanobeir_selection/nanobeir_pooling_sweep_20260219_191618",
     EVAL_DIR / "fiqa-pool-span-2/nanobeir_selection/nanobeir_pooling_sweep_20260218_192922",
     EVAL_DIR / "fiqa-pool-span-all/nanobeir_selection/nanobeir_pooling_sweep_20260219_181043",
     EVAL_DIR / "fiqa-pool-hier-all/nanobeir_selection/nanobeir_pooling_sweep_20260219_184252",
     EVAL_DIR / "fiqa-pool-kmeans-all/nanobeir_selection/nanobeir_pooling_sweep_20260219_181042",
]


# Set to None to keep all pool factors, or provide a list like [1, 2, 3].
POOL_FACTORS_TO_INCLUDE = [1, 2, 3, 4, 5, 6]

METRICS = ["ndcg@10", "mrr@10", "map@100", "recall@10"]
OUTPUT_PATH = EVAL_DIR / "combined_model_metrics.csv"
ROUND_DECIMALS = 4

In [ ]:
def normalize_method(pool_method_cli: Any, pool_method: Any, use_sklearn: bool) -> str:
    raw_value = pool_method_cli if pd.notna(pool_method_cli) else pool_method
    method = str(raw_value if raw_value is not None else "").strip().lower()
    mapping = {"slice": "span", "hier": "hierarchical"}
    method = mapping.get(method, method)
    if method == "kmeans" and bool(use_sklearn):
        method = "kmeans_sk"
    return method if method else "unknown"


def parse_scores(scores: dict[str, Any], metrics: list[str]) -> dict[str, dict[str, float]]:
    dataset_results: dict[str, dict[str, float]] = {}
    if not isinstance(scores, dict):
        return dataset_results

    # BEIR-style format: {dataset_name: {metric: value}}
    # Parse all dict-valued entries and ignore other auxiliary keys.
    if scores and any(isinstance(value, dict) for value in scores.values()):
        for dataset_name, dataset_metrics in scores.items():
            if not isinstance(dataset_metrics, dict):
                continue
            metric_row: dict[str, float] = {}
            for metric in metrics:
                if metric not in dataset_metrics:
                    continue
                try:
                    metric_row[metric] = float(dataset_metrics[metric])
                except (TypeError, ValueError):
                    metric_row[metric] = np.nan
            if metric_row:
                dataset_results[str(dataset_name)] = metric_row
        return dataset_results

    # NanoBEIR-style format: flat keys like NanoSciFact_MaxSim_ndcg@10
    for key, value in scores.items():
        if not isinstance(key, str) or "_MaxSim_" not in key:
            continue

        left, metric = key.split("_MaxSim_", 1)
        if metric not in metrics:
            continue
        if "mean" in left.lower():
            continue

        dataset_name = left.removeprefix("Nano")
        try:
            metric_value = float(value)
        except (TypeError, ValueError):
            metric_value = np.nan

        dataset_results.setdefault(dataset_name, {})[metric] = metric_value

    return dataset_results


def iter_result_payloads(results_dir: Path) -> list[tuple[Path, dict[str, Any]]]:
    payloads: list[tuple[Path, dict[str, Any]]] = []
    for json_path in sorted(results_dir.glob("*.json")):
        if json_path.name == "overview_summary.json":
            continue

        try:
            payload = json.loads(json_path.read_text())
        except json.JSONDecodeError:
            print(f"Skipping invalid JSON file: {json_path.name}")
            continue

        if not isinstance(payload, dict):
            continue
        if "scores" not in payload or "model" not in payload:
            continue

        payloads.append((json_path, payload))

    return payloads


def as_int_or_nan(value: Any) -> int | float:
    try:
        return int(value)
    except (TypeError, ValueError):
        return np.nan


def extract_model_name(model_path: str) -> str:
    normalized = str(model_path).replace("\\", "/").strip("/")
    parts = [part for part in normalized.split("/") if part]
    if len(parts) >= 2:
        return parts[-2]
    return str(model_path)


def build_per_model_table(
    payloads: list[tuple[Path, dict[str, Any]]],
    source_dir: Path,
    target_model_path: str,
    metrics: list[str],
    pool_factors_to_include: set[int] | None,
) -> pd.DataFrame:
    rows: list[dict[str, Any]] = []

    for json_path, payload in payloads:
        if str(payload.get("model")) != target_model_path:
            continue

        dataset_results = parse_scores(payload.get("scores", {}), metrics)
        method = normalize_method(
            payload.get("pool_method_cli"),
            payload.get("pool_method"),
            bool(payload.get("use_sklearn", False)),
        )
        pool_factor = as_int_or_nan(payload.get("pool_factor"))
        if pool_factors_to_include is not None:
            if pd.isna(pool_factor) or int(pool_factor) not in pool_factors_to_include:
                continue

        for dataset_name, metric_values in dataset_results.items():
            row = {
                "model_name": extract_model_name(target_model_path),
                "dataset": str(dataset_name),
                "method": method,
                "pool_factor": pool_factor,
            }
            for metric in metrics:
                row[metric] = metric_values.get(metric, np.nan)
            row["model_path"] = target_model_path
            row["results_dir"] = str(source_dir)
            row["results_json"] = json_path.name
            rows.append(row)

    out = pd.DataFrame(rows)
    ordered_cols = ["model_name", "dataset", "method", "pool_factor", *metrics, "model_path", "results_dir", "results_json"]

    if out.empty:
        return pd.DataFrame(columns=ordered_cols)

    out = (
        out[ordered_cols]
        .sort_values(["model_name", "dataset", "method", "pool_factor", "results_dir", "results_json"])
        .reset_index(drop=True)
    )
    return out

In [ ]:
pool_factor_set: set[int] | None = None
if POOL_FACTORS_TO_INCLUDE is not None:
    if isinstance(POOL_FACTORS_TO_INCLUDE, (int, float)):
        pool_factor_set = {int(POOL_FACTORS_TO_INCLUDE)}
    else:
        pool_factor_set = {int(value) for value in POOL_FACTORS_TO_INCLUDE}

normalized_sources: list[Path] = []
for source in RESULT_SOURCES:
    if not isinstance(source, (str, Path)):
        raise TypeError(f"Each entry in RESULT_SOURCES must be path-like. Got: {type(source)!r}")
    normalized_sources.append(Path(source))

if not normalized_sources:
    raise ValueError("RESULT_SOURCES is empty. Add at least one results directory.")

# Preserve input order while removing duplicates.
seen_sources: set[str] = set()
unique_sources: list[Path] = []
for source_path in normalized_sources:
    key = str(source_path)
    if key in seen_sources:
        continue
    seen_sources.add(key)
    unique_sources.append(source_path)

per_model_tables: list[pd.DataFrame] = []

for results_dir in unique_sources:
    if not results_dir.exists():
        raise FileNotFoundError(f"Results directory does not exist: {results_dir}")

    payloads = iter_result_payloads(results_dir)
    if not payloads:
        print(f"No usable JSON result files found in: {results_dir}. Skipping.")
        continue

    discovered_models = sorted({str(payload.get("model")) for _, payload in payloads})

    print("\nResults directory:", results_dir)
    print("Usable JSON files:", len(payloads))
    print("Discovered model paths:", discovered_models)
    if pool_factor_set is not None:
        print("Pool factors filter:", sorted(pool_factor_set))

    for model_path in discovered_models:
        model_df = build_per_model_table(payloads, results_dir, model_path, METRICS, pool_factor_set)
        if model_df.empty:
            print(f"No metric rows found for model path: {model_path} in {results_dir}")
            continue

        print(f"Per-model table for '{model_path}' from '{results_dir.name}': {len(model_df)} rows")
        per_model_tables.append(model_df)

if not per_model_tables:
    raise ValueError("No per-model tables were generated from RESULT_SOURCES.")

combined_metrics = (
    pd.concat(per_model_tables, ignore_index=True)
    .sort_values(["model_name", "dataset", "method", "pool_factor", "results_dir", "results_json"])
    .reset_index(drop=True)
)

OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
combined_metrics.to_csv(OUTPUT_PATH, index=False)

print(f"\nSaved combined table to: {OUTPUT_PATH}")
print("Combined rows:", len(combined_metrics))